In [60]:
remove.packages("rwwa")

Removing package from ‘/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/library’
(as ‘lib’ is unspecified)



In [62]:
# Installera RWWA kodpaket till R

devtools::install_github("WorldWeatherAttribution/rwwa")
library(rwwa)
library(extRemes)


── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/private/var/folders/x9/4rv4fwp54zq011_5p2wc8z9h0000gn/T/RtmpY8A6iV/remotesa3681858c54c/WorldWeatherAttribution-rwwa-ad29462/DESCRIPTION’ ... OK
* preparing ‘rwwa’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted ‘LazyData’ from DESCRIPTION
* building ‘rwwa_0.1.1.tar.gz’



* Ladda in datamängderna

In [58]:
gmst <- read.csv("/Users/joel/Livet/Globala System/År 3/LP 3/Kandidatarbete/Newell-Matthew-CCDetectonAttribution_Esbjerg-ac778b5/files/Rx1day-files/annual_sampling/observations/GMST/GMST_anomalies.csv", col.names = c("year", "GMST"), sep = ";", header = TRUE)
df_esbjerg <- read.csv("/Users/joel/Livet/Globala System/År 3/LP 3/Kandidatarbete/Newell-Matthew-CCDetectonAttribution_Esbjerg-ac778b5/files/Rx1day-files/annual_sampling/observations/annual_max_pr_station_annual.csv", col.names = c("year", "pr"), sep = ";", header = TRUE)

df_esbjerg1 <- merge(gmst, ts_station, by = "year", all = TRUE)
df_esbjerg2 <- df_station[df_station$year <= 2023, ] 

* Skapar stationära och ickestionära modell enligt WWA:s parametrisering

In [59]:
mdl_stat1 <- fit_ns(dist = "gev", type = "fixeddisp", data = df_esbjerg1, varnm = "pr", covnm = NA)
mdl1 <- fit_ns(dist="gev", type="fixeddisp", data=df_esbjerg1, varnm="pr", covnm=c("GMST")) #c("GMST")

mdl_stat2 <- fit_ns(dist = "gev", type = "fixeddisp", data = df_esbjerg2, varnm = "pr", covnm = NA)
mdl2 <- fit_ns(dist="gev", type="fixeddisp", data=df_esbjerg2, varnm="pr", covnm=c("GMST")) #c("GMST")

Warning message:
“restarting interrupted promise evaluation”
Warning message:
“internal error 1 in R_decompress1 with libdeflate”


ERROR: Error: lazy-load database '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/library/rwwa/R/rwwa.rdb' is corrupt


* Funktion för beräkning av p-värde

In [26]:
p_value <- function (mdl_stat,mdl_nonstat){
    
    logL_null <- -mdl_stat$value
    logL_alt  <- -mdl_nonstat$value # Din modell med covnm = "GMST"
    
    # Beräkna D
    D <- 2 * (logL_alt - logL_null)
    
    # Bestäm frihetsgrader (df)
    # Skillnaden är 1 parameter (alpha_GMST)
    df_diff <- 1 
    
    # Beräkna p-värdet
    p_val <- 1 - pchisq(D, df = df_diff)

    return (p_val)
    }


In [28]:
p_value(mdl_stat1, mdl1)
p_value(mdl_stat2, mdl2)

ERROR: Error: object 'mdl' not found


* Kör RWWA:s funktion boot_ci med tröskelvärde 144.6 för 2024 inkluderat och exkluderat

In [7]:
res_1 <- boot_ci(mdl,
                   cov_f  = cov_2024,
                   cov_cf = cov_1900,
                   ev     = 144.6,
                   nsamp  = 500)

res_2 <- boot_ci(mdl,
                   cov_f  = cov_2024,
                   cov_cf = cov_1900,
                   ev     = 144.6,
                   nsamp  = 500)

ERROR: Error: object 'cov_2024' not found
